# Live Virtual Try-On (Colab + GitHub Clone)

This notebook is built for your flow:
1. Clone repo from GitHub in Colab
2. Install deps
3. Launch live Gradio webcam UI


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
print('Colab:', IS_COLAB, '| Kaggle:', IS_KAGGLE)


## 1) Clone or update your GitHub repo

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Mothieram/tryon.git'   # change if needed
REPO_DIR_NAME = 'Tryon'
REPO_BRANCH = ''   # leave empty to auto-detect default branch

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
base = Path('/content') if IS_COLAB else (Path('/kaggle/working') if IS_KAGGLE else Path.cwd())
PROJECT_DIR = base / REPO_DIR_NAME

def run(cmd, cwd=None):
    print('>>', ' '.join(cmd))
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout.strip())
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr.strip())
        raise RuntimeError(f'Command failed ({p.returncode}): {' '.join(cmd)}')
    return p

if not REPO_BRANCH:
    probe = subprocess.run(['git','ls-remote','--symref',REPO_URL,'HEAD'], text=True, capture_output=True)
    if probe.returncode != 0:
        print(probe.stderr)
        raise RuntimeError('Could not read remote HEAD. Check REPO_URL and repo visibility.')
    branch_line = next((ln for ln in probe.stdout.splitlines() if ln.startswith('ref: ')), '')
    REPO_BRANCH = branch_line.split('refs/heads/')[-1].split('	')[0] if 'refs/heads/' in branch_line else 'main'

print('Using branch:', REPO_BRANCH)

if PROJECT_DIR.exists():
    run(['git','fetch','--all','--prune'], cwd=PROJECT_DIR)
    run(['git','checkout',REPO_BRANCH], cwd=PROJECT_DIR)
    run(['git','pull'], cwd=PROJECT_DIR)
else:
    run(['git','clone','--branch',REPO_BRANCH,REPO_URL,str(PROJECT_DIR)])

assert (PROJECT_DIR / 'engine').exists(), f'engine/ not found in {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Using PROJECT_DIR =', PROJECT_DIR)


## 2) Install dependencies

In [ ]:
%%bash
set -e
python -m pip install --upgrade pip
python -m pip install gradio opencv-contrib-python ultralytics scipy colorlog matplotlib imageio tqdm pyyaml psutil requests


## 3) Load pipeline

In [ ]:
import cv2
import numpy as np
import threading
import gradio as gr

from engine.render_pipeline import RenderPipeline

pipeline = RenderPipeline(
    pose_model='yolov8n-pose.pt',
    device='auto',
    target_fps=30,
    enable_shadows=True,
    enable_lighting=True,
    opacity=0.95,
)

ok = pipeline.load_models()
print('load_models:', ok)
if not ok:
    raise RuntimeError('Model loading failed')

count = pipeline.load_garments(str(PROJECT_DIR / 'assets' / 'shirts'))
shirt_names = pipeline.get_shirt_names()
print('garments loaded:', count)
print('shirts:', shirt_names)

pipe_lock = threading.Lock()


## 4) Launch live Gradio UI

In [ ]:
def _shirt_idx(name: str) -> int:
    try:
        return shirt_names.index(name)
    except Exception:
        return 0

def _status(stats) -> str:
    return (
        f'FPS: {stats.fps:.1f} | Pose: {stats.pose_detected} | '
        f'PoseMS: {stats.pose_ms:.1f} | WarpMS: {stats.warp_ms:.1f} | '
        f'Shirt: {stats.active_shirt}'
    )

def process_frame(frame_rgb, shirt_name, opacity, shadows, lighting):
    if frame_rgb is None:
        return None, 'Waiting for webcam frame...'

    frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
    with pipe_lock:
        pipeline.select_shirt(_shirt_idx(shirt_name))
        pipeline.opacity = float(opacity)
        pipeline.enable_shadows = bool(shadows)
        pipeline.enable_lighting = bool(lighting)
        out_bgr, stats = pipeline.process_frame(frame_bgr)

    out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
    return out_rgb, _status(stats)

def next_shirt(current_name):
    if not shirt_names:
        return gr.update(value='shirt0'), 'No shirts loaded'
    i = (_shirt_idx(current_name) + 1) % len(shirt_names)
    return gr.update(value=shirt_names[i]), f'Selected: {shirt_names[i]}'

def prev_shirt(current_name):
    if not shirt_names:
        return gr.update(value='shirt0'), 'No shirts loaded'
    i = (_shirt_idx(current_name) - 1) % len(shirt_names)
    return gr.update(value=shirt_names[i]), f'Selected: {shirt_names[i]}'

with gr.Blocks(title='Virtual Try-On Live') as demo:
    gr.Markdown('## Live Webcam Virtual Try-On')
    gr.Markdown('Allow camera, then it auto-runs live warping (like app.py/main.py).')

    with gr.Row():
        cam = gr.Image(label='Webcam Input', sources=['webcam'], type='numpy')
        out = gr.Image(label='Try-On Output', type='numpy')

    with gr.Row():
        prev_btn = gr.Button('◀ Previous Shirt')
        next_btn = gr.Button('Next Shirt ▶')

    with gr.Row():
        shirt = gr.Dropdown(
            choices=shirt_names if shirt_names else ['shirt0'],
            value=(shirt_names[0] if shirt_names else 'shirt0'),
            label='Shirt',
        )
        opac = gr.Slider(0.2, 1.0, value=0.95, step=0.01, label='Opacity')
        shadows = gr.Checkbox(value=True, label='Shadows')
        lighting = gr.Checkbox(value=True, label='Lighting')

    status = gr.Textbox(label='Status', interactive=False, value='Waiting for webcam...')
    run_btn = gr.Button('Process One Frame (Manual)')

    prev_btn.click(prev_shirt, [shirt], [shirt, status], queue=False)
    next_btn.click(next_shirt, [shirt], [shirt, status], queue=False)

    # Live stream processing: camera frames continuously call process_frame.
    cam.stream(
        process_frame,
        [cam, shirt, opac, shadows, lighting],
        [out, status],
        stream_every=0.03,
        time_limit=None,
    )

    # Fallback/manual triggers.
    cam.change(process_frame, [cam, shirt, opac, shadows, lighting], [out, status], queue=False)
    run_btn.click(process_frame, [cam, shirt, opac, shadows, lighting], [out, status], queue=False)

share_flag = True if IS_COLAB else False
demo.launch(share=share_flag, show_error=True)


In [ ]:
# Optional cleanup
# pipeline.unload_models()
